# LSEG Workspace — Assessment 2: Quantitative Risk Analysis
**Individual Reflective Report | Risk Management & Strategy Backtesting**

---
### Contents
1. Setup & Data Loading
2. Return Distribution & Descriptive Statistics
3. Value at Risk (VaR) — Historical, Parametric, Monte Carlo
4. Expected Shortfall (CVaR)
5. Drawdown Analysis
6. Momentum Strategy Backtest
7. Position Sizing — Kelly Criterion
8. ATR-Based Stop-Loss
9. Put Option Hedging Simulation
10. Summary Risk Dashboard

---
> **Data source:** LSEG `refinitiv.data` (primary) → `yfinance` (fallback) → calibrated synthetic data (offline fallback)
> 
> **Instruments:** NVDA.O, AAPL.O, EUR/USD, GBP/USD

## Section 1 — Setup & Data Loading

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Plot Style ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
    'font.family':      'monospace',
    'figure.dpi':       110,
})
BLUE   = '#58a6ff'
GREEN  = '#3fb950'
RED    = '#f85149'
YELLOW = '#e3b341'
PURPLE = '#bc8cff'
ORANGE = '#ffa657'
print('Imports OK')

In [ ]:
# ── Data Loading ─────────────────────────────────────────────────────────
# Priority: (1) LSEG refinitiv.data  (2) yfinance  (3) calibrated synthetic

try:
    import refinitiv.data as rd
    rd.open_session()
    raw = rd.get_history(
        universe=['NVDA.O', 'AAPL.O', 'EUR=', 'GBP='],
        fields=['TRDPRC_1'],
        interval='1D',
        count=504
    )
    prices = raw['TRDPRC_1'].rename(columns={
        'NVDA.O': 'NVDA', 'AAPL.O': 'AAPL',
        'EUR=': 'EURUSD', 'GBP=': 'GBPUSD'
    })
    print('[LSEG] Data loaded via refinitiv.data')

except Exception:
    try:
        import yfinance as yf
        tickers = {'NVDA': 'NVDA', 'AAPL': 'AAPL',
                   'EURUSD': 'EURUSD=X', 'GBPUSD': 'GBPUSD=X'}
        frames = {}
        for name, yticker in tickers.items():
            df = yf.download(yticker, period='2y', interval='1d',
                             auto_adjust=True, progress=False)
            frames[name] = df['Close']
        prices = pd.DataFrame(frames).dropna()
        print('[yfinance] Data loaded')

    except Exception:
        print('[Synthetic] Generating LSEG-calibrated data (no live feed)...')
        np.random.seed(42)
        n = 504
        dates = pd.bdate_range('2023-01-02', periods=n)
        params = {
            'NVDA':   (0.00120, 0.0280),
            'AAPL':   (0.00060, 0.0175),
            'EURUSD': (0.00004, 0.0045),
            'GBPUSD': (0.00005, 0.0048),
        }
        cols = list(params.keys())
        mus  = np.array([params[c][0] for c in cols])
        sigs = np.array([params[c][1] for c in cols])
        corr = np.array([
            [1.00, 0.60, 0.10, 0.12],
            [0.60, 1.00, 0.12, 0.14],
            [0.10, 0.12, 1.00, 0.72],
            [0.12, 0.14, 0.72, 1.00],
        ])
        cov = np.outer(sigs, sigs) * corr
        raw_norm = np.random.multivariate_normal(np.zeros(4), cov, n)
        dof = 6
        t_scale = np.random.chisquare(dof, (n, 1)) / dof
        ret_arr = mus + raw_norm / np.sqrt(t_scale)
        ret_df  = pd.DataFrame(ret_arr, index=dates, columns=cols)
        start   = {'NVDA': 150.0, 'AAPL': 130.0, 'EURUSD': 1.07, 'GBPUSD': 1.23}
        prices  = pd.DataFrame(
            {c: start[c] * (1 + ret_df[c]).cumprod() for c in cols}, index=dates
        )
        print('[Synthetic] Done — 504 trading days, 4 instruments')

prices  = prices.dropna()
returns = prices.pct_change().dropna()
log_ret = np.log(prices / prices.shift(1)).dropna()

print(f'\nDate range   : {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'Observations : {len(returns)} trading days')
print(f'Instruments  : {list(prices.columns)}')
prices.tail()

## Section 2 — Return Distribution & Descriptive Statistics

In [ ]:
# ── Descriptive Statistics ───────────────────────────────────────────────
desc = returns.describe().T
desc['skewness']  = returns.skew()
desc['kurtosis']  = returns.kurtosis()
desc['ann_return'] = returns.mean() * 252
desc['ann_vol']    = returns.std()  * np.sqrt(252)
desc['sharpe']     = desc['ann_return'] / desc['ann_vol']
desc[['mean','std','skewness','kurtosis','ann_return','ann_vol','sharpe']].round(4)

In [ ]:
# ── Fig 1: Return Distribution ───────────────────────────────────────────
nvda_ret = returns['NVDA'].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('NVDA Daily Return Distribution', fontsize=14, color=BLUE, fontweight='bold')

ax = axes[0]
ax.hist(nvda_ret, bins=60, color=BLUE, alpha=0.7, edgecolor='none', density=True)
x = np.linspace(nvda_ret.min(), nvda_ret.max(), 300)
ax.plot(x, stats.norm.pdf(x, nvda_ret.mean(), nvda_ret.std()),
        color=YELLOW, lw=2, label='Normal Fit')
ax.set_title('Histogram + Normal Fit', color='#c9d1d9')
ax.set_xlabel('Daily Return')
ax.legend()

ax2 = axes[1]
stats.probplot(nvda_ret, dist='norm', plot=ax2)
ax2.get_lines()[0].set(color=BLUE, alpha=0.6, markersize=3)
ax2.get_lines()[1].set(color=RED, lw=2)
ax2.set_title('Q-Q Plot (vs Normal)', color='#c9d1d9')

plt.tight_layout()
plt.savefig('fig1_return_distribution.png', bbox_inches='tight')
plt.show()

## Section 3 — Value at Risk (VaR)

In [ ]:
# ── VaR Functions ────────────────────────────────────────────────────────
POSITION    = 100_000
CONF_LEVELS = [0.90, 0.95, 0.99]

def historical_var(returns, confidence):
    return -np.percentile(returns, (1 - confidence) * 100)

def parametric_var(returns, confidence):
    mu, sigma = returns.mean(), returns.std()
    z = stats.norm.ppf(1 - confidence)
    return -(mu + z * sigma)

def mc_var(returns, confidence, n_sims=100_000):
    mu, sigma = returns.mean(), returns.std()
    sim = np.random.normal(mu, sigma, n_sims)
    return -np.percentile(sim, (1 - confidence) * 100)

var_results = []
for c in CONF_LEVELS:
    hvar = historical_var(nvda_ret, c)
    pvar = parametric_var(nvda_ret, c)
    mvar = mc_var(nvda_ret, c)
    var_results.append({
        'Confidence':  f'{c:.0%}',
        'Hist VaR (%)':  f'{hvar:.2%}',
        'Para VaR (%)':  f'{pvar:.2%}',
        'MC VaR (%)':    f'{mvar:.2%}',
        'Hist VaR (£)':  f'£{hvar*POSITION:,.0f}',
        'Para VaR (£)':  f'£{pvar*POSITION:,.0f}',
        'MC VaR (£)':    f'£{mvar*POSITION:,.0f}',
    })

pd.DataFrame(var_results)

In [ ]:
# ── Fig 2: VaR Distribution Plot ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('NVDA VaR — Return Distribution with Risk Thresholds',
             fontsize=13, color=BLUE, fontweight='bold')

ax.hist(nvda_ret, bins=80, density=True, color=BLUE, alpha=0.5, edgecolor='none',
        label='Daily Returns')
x = np.linspace(nvda_ret.min(), nvda_ret.max(), 300)
ax.plot(x, stats.norm.pdf(x, nvda_ret.mean(), nvda_ret.std()),
        color=YELLOW, lw=2, label='Normal Fit')

colors_var = [GREEN, ORANGE, RED]
for row, col in zip(var_results, colors_var):
    v = -float(row['Hist VaR (%)'].strip('%')) / 100
    ax.axvline(v, color=col, lw=2, linestyle='--',
               label=f"VaR {row['Confidence']} = {row['Hist VaR (%)']}")

ax.fill_betweenx([0, 15], nvda_ret.min(),
                 -float(var_results[2]['Hist VaR (%)'].strip('%')) / 100,
                 color=RED, alpha=0.12, label='99% Tail Region')
ax.set_xlabel('Daily Return')
ax.set_ylabel('Density')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig2_var_distribution.png', bbox_inches='tight')
plt.show()

## Section 4 — Expected Shortfall (CVaR)

In [ ]:
# ── CVaR Calculation ─────────────────────────────────────────────────────
def cvar(returns, confidence):
    var_thresh = -np.percentile(returns, (1 - confidence) * 100)
    tail = returns[returns <= -var_thresh]
    return -tail.mean() if len(tail) > 0 else var_thresh

cvar_results = []
for c in CONF_LEVELS:
    hvar = historical_var(nvda_ret, c)
    cv   = cvar(nvda_ret, c)
    cvar_results.append({
        'Confidence':      f'{c:.0%}',
        'VaR (%)':         f'{hvar:.2%}',
        'CVaR (%)':        f'{cv:.2%}',
        'VaR (£)':         f'£{hvar*POSITION:,.0f}',
        'CVaR (£)':        f'£{cv*POSITION:,.0f}',
        'CVaR/VaR ratio':  f'{cv/hvar:.2f}x'
    })

pd.DataFrame(cvar_results)

In [ ]:
# ── Fig 3: CVaR Tail Illustration ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('NVDA Expected Shortfall (CVaR) — 99% Confidence',
             fontsize=13, color=BLUE, fontweight='bold')

ax.hist(nvda_ret, bins=80, density=True, color=BLUE, alpha=0.5, edgecolor='none')
var99 = historical_var(nvda_ret, 0.99)
cv99  = cvar(nvda_ret, 0.99)

ax.axvline(-var99, color=ORANGE, lw=2.5, linestyle='--',
           label=f'VaR 99%  = {-var99:.2%}')
ax.axvline(-cv99,  color=RED,    lw=2.5, linestyle='-',
           label=f'CVaR 99% = {-cv99:.2%}')

tail_vals = nvda_ret[nvda_ret <= -var99]
if len(tail_vals):
    ax.fill_betweenx(
        [0, stats.norm.pdf(-var99, nvda_ret.mean(), nvda_ret.std()) * 0.8],
        nvda_ret.min(), -var99,
        color=RED, alpha=0.25, label='CVaR Tail Region'
    )
ax.set_xlabel('Daily Return')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig('fig3_cvar.png', bbox_inches='tight')
plt.show()

## Section 5 — Drawdown Analysis

In [ ]:
# ── Drawdown Metrics ─────────────────────────────────────────────────────
cum_ret  = (1 + returns['NVDA']).cumprod()
roll_max = cum_ret.cummax()
drawdown = (cum_ret - roll_max) / roll_max

max_dd      = drawdown.min()
max_dd_date = drawdown.idxmin()

dd_periods = []
start = None
for date, val in drawdown.items():
    if val < 0 and start is None:
        start = date
    elif val >= 0 and start is not None:
        dd_periods.append((start, date, (date - start).days))
        start = None

avg_recovery = np.mean([d[2] for d in dd_periods]) if dd_periods else 0
calmar       = returns['NVDA'].mean() * 252 / abs(max_dd)

summary_dd = pd.DataFrame({
    'Metric': ['Max Drawdown', 'Max DD Date', 'DD Periods', 'Avg Recovery (days)', 'Calmar Ratio'],
    'Value':  [f'{max_dd:.2%}', str(max_dd_date.date()), len(dd_periods),
               f'{avg_recovery:.1f}', f'{calmar:.2f}']
})
summary_dd

In [ ]:
# ── Fig 4: Equity Curve + Drawdown ───────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})
fig.suptitle('NVDA — Cumulative Return & Drawdown', fontsize=13, color=BLUE, fontweight='bold')

ax1.plot(cum_ret.index, cum_ret.values, color=BLUE, lw=1.5, label='Cumulative Return')
ax1.plot(roll_max.index, roll_max.values, color=YELLOW, lw=1, linestyle='--',
         alpha=0.6, label='Rolling Maximum')
ax1.set_ylabel('Cumulative Return')
ax1.legend()
ax1.set_title('Equity Curve', color='#c9d1d9', fontsize=11)

ax2.fill_between(drawdown.index, drawdown.values, 0, color=RED, alpha=0.4)
ax2.plot(drawdown.index, drawdown.values, color=RED, lw=1)
ax2.axhline(max_dd, color=ORANGE, lw=1.5, linestyle='--',
            label=f'Max DD = {max_dd:.2%}')
ax2.set_ylabel('Drawdown')
ax2.set_xlabel('Date')
ax2.legend()
ax2.set_title('Drawdown', color='#c9d1d9', fontsize=11)

plt.tight_layout()
plt.savefig('fig4_drawdown.png', bbox_inches='tight')
plt.show()

## Section 6 — Momentum Strategy Backtest

In [ ]:
# ── Momentum Strategy ────────────────────────────────────────────────────
def momentum_backtest(price_series, lookback=20):
    """Go long when 20-day momentum > 0, else flat."""
    px       = price_series.copy()
    momentum = px.pct_change(lookback).shift(1)
    signal   = (momentum > 0).astype(int)
    return (signal * px.pct_change()).dropna()

def performance_metrics(ret, label='Strategy'):
    ann_ret  = ret.mean() * 252
    ann_vol  = ret.std()  * np.sqrt(252)
    sharpe   = ann_ret / ann_vol
    cum      = (1 + ret).cumprod()
    max_dd   = ((cum / cum.cummax()) - 1).min()
    calmar   = ann_ret / abs(max_dd)
    win_rate = (ret > 0).mean()
    return {
        'Label':      label,
        'Ann Return': f'{ann_ret:.2%}',
        'Ann Vol':    f'{ann_vol:.2%}',
        'Sharpe':     f'{sharpe:.2f}',
        'Max DD':     f'{max_dd:.2%}',
        'Calmar':     f'{calmar:.2f}',
        'Win Rate':   f'{win_rate:.2%}',
    }

strat_ret  = momentum_backtest(prices['NVDA'])
market_ret = returns['NVDA'].loc[strat_ret.index]

pd.DataFrame([
    performance_metrics(strat_ret,  'Momentum Strategy'),
    performance_metrics(market_ret, 'Buy & Hold NVDA')
])

In [ ]:
# ── Fig 5: Strategy vs Buy & Hold ────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})
fig.suptitle('Momentum Strategy vs Buy & Hold — NVDA',
             fontsize=13, color=BLUE, fontweight='bold')

cum_strat  = (1 + strat_ret).cumprod()
cum_market = (1 + market_ret).cumprod()

ax1.plot(cum_strat.index,  cum_strat.values,  color=GREEN, lw=1.5, label='Momentum Strategy')
ax1.plot(cum_market.index, cum_market.values, color=BLUE,  lw=1.5, label='Buy & Hold', alpha=0.7)
ax1.set_ylabel('Cumulative Return')
ax1.legend()
ax1.set_title('Equity Curves', color='#c9d1d9')

strat_dd = (cum_strat / cum_strat.cummax() - 1)
ax2.fill_between(strat_dd.index, strat_dd.values, 0, color=RED, alpha=0.4, label='Strategy DD')
ax2.set_ylabel('Drawdown')
ax2.set_xlabel('Date')
ax2.legend()
ax2.set_title('Strategy Drawdown', color='#c9d1d9')

plt.tight_layout()
plt.savefig('fig5_momentum_backtest.png', bbox_inches='tight')
plt.show()

## Section 7 — Position Sizing: Kelly Criterion

In [ ]:
# ── Kelly Criterion ──────────────────────────────────────────────────────
win_rate      = (strat_ret > 0).mean()
avg_win       = strat_ret[strat_ret > 0].mean()
avg_loss      = abs(strat_ret[strat_ret < 0].mean())
win_loss_ratio = avg_win / avg_loss

kelly_full = win_rate - (1 - win_rate) / win_loss_ratio
kelly_half = kelly_full * 0.5
kelly_cap  = 0.05  # Hard cap at 5%

pd.DataFrame({
    'Metric': ['Win Rate', 'Avg Win', 'Avg Loss', 'Win/Loss Ratio',
               'Full Kelly', 'Half Kelly', 'Position Cap (used)'],
    'Value':  [f'{win_rate:.2%}', f'{avg_win:.4f}', f'{avg_loss:.4f}',
               f'{win_loss_ratio:.2f}', f'{kelly_full:.2%}',
               f'{kelly_half:.2%}', '5.00%']
})

In [ ]:
# ── Fig 6: Kelly Growth Curve ────────────────────────────────────────────
fractions = np.linspace(0, 1, 200)
growth    = [np.mean(np.log(1 + f * strat_ret)) for f in fractions]

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('Kelly Criterion — Expected Log Growth vs Position Fraction',
             fontsize=13, color=BLUE, fontweight='bold')
ax.plot(fractions, growth, color=BLUE, lw=2)
ax.axvline(max(kelly_full, 0), color=RED,    lw=2, linestyle='--',
           label=f'Full Kelly = {kelly_full:.2%}')
ax.axvline(max(kelly_half, 0), color=GREEN,  lw=2, linestyle='--',
           label=f'Half Kelly = {kelly_half:.2%}')
ax.axvline(kelly_cap,          color=YELLOW, lw=2, linestyle=':',
           label='5% Cap (used)')
ax.axhline(0, color='#8b949e', lw=1)
ax.set_xlabel('Position Fraction of Portfolio')
ax.set_ylabel('Expected Log Growth')
ax.legend()
plt.tight_layout()
plt.savefig('fig6_kelly.png', bbox_inches='tight')
plt.show()

## Section 8 — ATR-Based Stop-Loss

In [ ]:
# ── ATR Calculation ──────────────────────────────────────────────────────
def compute_atr(price_series, period=14):
    hi  = price_series * 1.005
    lo  = price_series * 0.995
    tr  = hi - lo
    return tr.rolling(period).mean()

atr        = compute_atr(prices['NVDA'])
atr_mult   = 2.0
last_price = prices['NVDA'].iloc[-1]
last_atr   = atr.iloc[-1]
stop_price = last_price - atr_mult * last_atr
stop_pct   = (last_price - stop_price) / last_price

pd.DataFrame({
    'Metric': ['Last Price', 'ATR (14-day)', 'ATR Multiplier',
               'Stop-Loss Price', 'Stop-Loss Distance (%)'],
    'Value':  [f'${last_price:.2f}', f'${last_atr:.2f}', f'{atr_mult}x',
               f'${stop_price:.2f}', f'{stop_pct:.2%}']
})

In [ ]:
# ── Fig 7: ATR Stop-Loss Overlay ─────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})
fig.suptitle('NVDA Price with ATR-Based Stop-Loss Band',
             fontsize=13, color=BLUE, fontweight='bold')

stop_series = prices['NVDA'] - atr_mult * atr

ax1.plot(prices['NVDA'].index, prices['NVDA'].values, color=BLUE, lw=1.5, label='NVDA Price')
ax1.plot(stop_series.index, stop_series.values, color=RED, lw=1.2,
         linestyle='--', label=f'2x ATR Stop')
ax1.fill_between(prices['NVDA'].index, prices['NVDA'].values,
                 stop_series.values, alpha=0.08, color=RED)
ax1.set_ylabel('Price (USD)')
ax1.legend()

ax2.plot(atr.index, atr.values, color=ORANGE, lw=1.2, label='ATR (14-day)')
ax2.set_ylabel('ATR')
ax2.set_xlabel('Date')
ax2.legend()

plt.tight_layout()
plt.savefig('fig7_atr_stop.png', bbox_inches='tight')
plt.show()

## Section 9 — Put Option Hedging Simulation

In [ ]:
# ── Hedging Simulation ───────────────────────────────────────────────────
PUT_COST_PCT   = 0.012   # 1.2% monthly premium for 5% OTM put
PUT_STRIKE_PCT = 0.95    # 5% OTM
REBALANCE_DAYS = 21      # Monthly rebalance

def hedged_returns(strategy_ret, put_cost=PUT_COST_PCT,
                   strike_pct=PUT_STRIKE_PCT, rebalance_days=REBALANCE_DAYS):
    hedged = []
    for i, (date, ret) in enumerate(strategy_ret.items()):
        if i % rebalance_days == 0:
            net = ret - put_cost / rebalance_days
        else:
            if ret < -(1 - strike_pct):
                net = max(ret, -(1 - strike_pct))
            else:
                net = ret
        hedged.append(net)
    return pd.Series(hedged, index=strategy_ret.index)

hedged_ret = hedged_returns(strat_ret)

pd.DataFrame([
    performance_metrics(strat_ret,  'Unhedged Strategy'),
    performance_metrics(hedged_ret, 'Hedged (5% OTM Put)')
])

In [ ]:
# ── Fig 8: Hedged vs Unhedged ─────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})
fig.suptitle('Hedged vs Unhedged Momentum Strategy — NVDA',
             fontsize=13, color=BLUE, fontweight='bold')

cum_unhedged = (1 + strat_ret).cumprod()
cum_hedged   = (1 + hedged_ret).cumprod()

ax1.plot(cum_unhedged.index, cum_unhedged.values, color=RED,   lw=1.5, label='Unhedged')
ax1.plot(cum_hedged.index,   cum_hedged.values,   color=GREEN, lw=1.5, label='Hedged (5% OTM Put)')
ax1.set_ylabel('Cumulative Return')
ax1.legend()

uh_dd = (cum_unhedged / cum_unhedged.cummax() - 1)
hd_dd = (cum_hedged   / cum_hedged.cummax()   - 1)
ax2.fill_between(uh_dd.index, uh_dd.values, 0, color=RED,   alpha=0.3, label='Unhedged DD')
ax2.fill_between(hd_dd.index, hd_dd.values, 0, color=GREEN, alpha=0.3, label='Hedged DD')
ax2.set_ylabel('Drawdown')
ax2.set_xlabel('Date')
ax2.legend()

plt.tight_layout()
plt.savefig('fig8_hedging.png', bbox_inches='tight')
plt.show()

## Section 10 — Summary Risk Dashboard

In [ ]:
# ── Full Risk Summary Table ──────────────────────────────────────────────
strat_p  = performance_metrics(strat_ret,  'Momentum Strategy')
hedged_p = performance_metrics(hedged_ret, 'Hedged Strategy')

summary = pd.DataFrame({
    'Metric': [
        'Ann. Return (Strategy)', 'Ann. Volatility', 'Sharpe Ratio',
        'Max Drawdown (Unhedged)', 'Max Drawdown (Hedged)', 'Calmar Ratio',
        'VaR 95% (1-day, Hist.)', 'VaR 99% (1-day, Hist.)', 'CVaR 99% (1-day)',
        'Position Cap', 'ATR Stop Distance', 'Deployment Rec.'
    ],
    'Value': [
        strat_p['Ann Return'], strat_p['Ann Vol'], strat_p['Sharpe'],
        strat_p['Max DD'], hedged_p['Max DD'], strat_p['Calmar'],
        f"{historical_var(nvda_ret, 0.95):.2%}",
        f"{historical_var(nvda_ret, 0.99):.2%}",
        f"{cvar(nvda_ret, 0.99):.2%}",
        '5% of portfolio', f'{stop_pct:.2%}', 'Paper-trade first'
    ]
})
summary

In [ ]:
# ── Fig 9: Summary Risk Bar Chart ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle('Risk & Performance Summary — NVDA Momentum Strategy',
             fontsize=13, color=BLUE, fontweight='bold')

metrics_plot = {
    'Sharpe':         float(strat_p['Sharpe']),
    'Calmar':         float(strat_p['Calmar']),
    'VaR 95%':        historical_var(nvda_ret, 0.95) * 100,
    'VaR 99%':        historical_var(nvda_ret, 0.99) * 100,
    'CVaR 99%':       cvar(nvda_ret, 0.99) * 100,
    'Max DD\n(Unhedged)': abs(float(strat_p['Max DD'].strip('%'))),
    'Max DD\n(Hedged)':   abs(float(hedged_p['Max DD'].strip('%'))),
}

bar_colors = [GREEN if k in ['Sharpe','Calmar'] else RED
              for k in metrics_plot]
bars = ax.bar(metrics_plot.keys(), metrics_plot.values(),
              color=bar_colors, alpha=0.8, edgecolor='none', width=0.5)

for bar, val in zip(bars, metrics_plot.values()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3, f'{val:.2f}',
            ha='center', va='bottom', color='#c9d1d9', fontsize=9)

ax.set_ylabel('Value (% or ratio)')
ax.axhline(0, color='#8b949e', lw=0.8)
plt.tight_layout()
plt.savefig('fig9_summary_risk.png', bbox_inches='tight')
plt.show()
print('\n All analysis complete!')